# Hierarchical Clustering

**ifri-mini-ml-lib · Module `clustering` · `HierarchicalClustering`**

Hierarchical clustering groups points into nested levels instead of a flat partition. It is useful when you want to understand the structure of the data before choosing the final number of clusters.

<img src="../assets/imgs/hierarchical_clustering/hierarchical_clustering_hero.svg" alt="Hierarchical clustering overview"/>

*Figure 1: Merge points first, then read the hierarchy as a dendrogram.*

## 1. Theory

Hierarchical clustering builds nested groups instead of a flat partition. It is useful when you want to inspect the structure of the data before choosing how many clusters to keep.

---

### Two opposite strategies

| | **Agglomerative** *(bottom-up)* | **Divisive** *(top-down)* |
|---|---|---|
| **Starting point** | Each point is one cluster | All points are in one cluster |
| **Operation** | Merge the two closest clusters | Split the largest cluster |
| **Stopping point** | One cluster, or the requested number of clusters | The requested number of clusters |
| **Complexity** | $O(n^3)$ naive, $O(n^2 \log n)$ optimized | $O(n^2 k)$ with bisection |
| **Typical use** | Very common | Less common |

In `ifri_mini_ml_lib`, agglomerative clustering uses exact pairwise distances, while divisive clustering uses K-Means with $k=2$ to bisect clusters.

## 2. Step-by-step algorithm

### Agglomerative algorithm

Let $\mathcal{X} = \{x_1, x_2, \ldots, x_n\}$ be a dataset with $n$ points.

**Initialization:** create $n$ singleton clusters: $C_1 = \{x_1\}, C_2 = \{x_2\}, \ldots, C_n = \{x_n\}$.

**Iteration:** at each step $t$:

$$
(C_i^*, C_j^*) = \arg\min_{i \neq j} \ d_{\text{link}}(C_i, C_j)
$$

Merge the closest clusters: $C_{\text{new}} = C_i^* \cup C_j^*$, then remove $C_i^*$ and $C_j^*$ from the active cluster list.

**Stopping rule:** stop when $k$ clusters remain, or continue to one cluster to build a complete dendrogram.

---

### Small example with 6 points

```
Step 0: {A} {B} {C} {D} {E} {F}   (6 clusters)
Step 1: {A,B} {C} {D} {E} {F}     (merge A and B: closest pair)
Step 2: {A,B} {C} {D,E} {F}       (merge D and E)
Step 3: {A,B,C} {D,E} {F}         (merge {A,B} and C)
Step 4: {A,B,C} {D,E,F}           (merge {D,E} and F) -> 2 clusters
```


## 3. Linkage criteria

The linkage function controls how the distance between two clusters is computed. It has a direct impact on the shape of the hierarchy.

Let $C_i$ and $C_j$ be two clusters. The Euclidean distance between two points is:

$d(x, y) = \|x - y\|_2$

---

### Single linkage (minimum)

$$d_{\text{single}}(C_i, C_j) = \min_{x \in C_i,\ y \in C_j} d(x, y)$$

Merges clusters using their closest points. It can create chain-like structures.

---

### Complete linkage (maximum)

$$d_{\text{complete}}(C_i, C_j) = \max_{x \in C_i,\ y \in C_j} d(x, y)$$

Uses the farthest points. It usually produces compact clusters.

---

### Average linkage (UPGMA)

$$d_{\text{average}}(C_i, C_j) = \frac{1}{|C_i| \cdot |C_j|} \sum_{x \in C_i} \sum_{y \in C_j} d(x, y)$$

Uses the average of all pairwise distances. It is often the best default.

<img src="../assets/imgs/hierarchical_clustering/hierarchical_clustering_linkage.svg" alt="Linkage comparison"/>

*Figure 2: Linkage changes the shape of the hierarchy.*

## 4. Interactive demo with Iris


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from ifri_mini_ml_lib.clustering import HierarchicalClustering
from ifri_mini_ml_lib.metrics.clustering import calculate_silhouette
from notebooks.clustering.utils import load_hierarchical_iris_data

In [ ]:
# Load the Iris dataset with the shared helper
iris = load_hierarchical_iris_data()
X = iris['data']
y_true = iris['target']

# Keep the first two features for 2D visualization
X_2d = X[:, :2]

print(f"Iris dataset loaded : {X.shape[0]} samples, {X.shape[1]} features")
print(f"True classes : {iris['target_names']}")

## 5. Dendrogram

A **dendrogram** visualizes the hierarchy produced by agglomerative clustering. A horizontal cut selects the number of clusters.

### How to read it

- **Horizontal axis**: samples and merged clusters
- **Vertical axis**: merge distance
- **Cut height**: higher cut, fewer clusters

> Look for the largest vertical gap when choosing a cut height.

In [ ]:
# Interactive dendrogram
w_dend_linkage = widgets.ToggleButtons(
    options=['single', 'complete', 'average'],
    value='complete',
    description='Linkage :',
    button_style='warning',
    style={'description_width': 'initial'},
)

w_dend_k = widgets.IntSlider(
    value=3, min=2, max=6, step=1,
    description='k (colored clusters):',
    style={'description_width': 'initial'},
)

w_dend_n = widgets.IntSlider(
    value=30, min=10, max=150, step=10,
    description='Samples used:',
    style={'description_width': 'initial'},
)

out_dend = widgets.Output()

def run_dend(change=None):
    with out_dend:
        clear_output(wait=True)
        n = w_dend_n.value
        linkage = w_dend_linkage.value
        k = w_dend_k.value

        # Stratified subsample to keep the 3 classes
        np.random.seed(42)
        idx = []
        for c in range(3):
            class_idx = np.where(y_true == c)[0]
            idx.extend(np.random.choice(class_idx, size=n // 3, replace=False))
        idx = np.array(idx)
        X_sub = X[:, [2, 3]][idx]
        y_sub = y_true[idx]

        hc = HierarchicalClustering(n_clusters=k, linkage=linkage)
        hc.fit_predict(X_sub)

        # Use the library dendrogram plotter
        species_labels = [iris['target_names'][y_sub[i]] for i in range(len(idx))]
        plt.figure(figsize=(14, 5))
        hc.plot_dendrogram(labels=species_labels)

for w in [w_dend_linkage, w_dend_k, w_dend_n]:
    w.observe(run_dend, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>Interactive dendrogram</h4>'),
    w_dend_linkage, w_dend_k, w_dend_n,
    out_dend
]))
run_dend()

## 6. Agglomerative vs Divisive


In [ ]:
import time

from notebooks.clustering.utils import plot_hierarchical_method_comparison

# Side-by-side comparison on the same subsample
np.random.seed(0)
idx50 = np.random.choice(len(X), 50, replace=False)
X50 = X[:, [2, 3]][idx50]
y50 = y_true[idx50]

results = []
for method, title in zip(
    ['agglomerative', 'divisive'],
    ['Agglomerative (bottom-up)', 'Divisive (top-down)']
):
    t0 = time.time()
    hc = HierarchicalClustering(n_clusters=3, linkage='complete', method=method)
    labels = hc.fit_predict(X50)
    elapsed = time.time() - t0
    sil = calculate_silhouette(X50, labels)

    results.append({
        'X': X50,
        'labels': labels,
        'title': title,
        'silhouette': sil,
        'elapsed': elapsed,
        'xlabel': 'Petal length (cm)',
        'ylabel': 'Petal width (cm)',
    })

plot_hierarchical_method_comparison(results)

## 7. Advantages and limitations

| | Advantages | Limitations |
|---|---|---|
| **No fixed k required upfront** | The dendrogram can guide the choice of $k$ | The dendrogram can be difficult to read on large datasets |
| **Deterministic** | Agglomerative clustering gives reproducible results | $O(n^3)$ time and $O(n^2)$ memory can be slow for large datasets |
| **Dendrogram** | Provides a full hierarchical view of the data structure | Results are sensitive to the linkage criterion |
| **Flexible cluster shapes** | Single linkage can detect non-convex structures | There is no centroid representation, which can make interpretation harder |

---

### When to use hierarchical clustering

- When the number of clusters is not known in advance
- When the dataset is small to medium-sized
- When a hierarchical view of the data is useful, such as taxonomy or phylogeny
- For data exploration and visualization

### When to prefer K-Means

- Large datasets
- Roughly spherical clusters of similar size
- Situations where fast execution is a priority


## 8. Practical usage

### Simple example: quick start

Here is how to use hierarchical clustering in a few lines:


In [ ]:
from ifri_mini_ml_lib.clustering import HierarchicalClustering
from sklearn.datasets import make_blobs
import numpy as np

# 1. Generate synthetic data
X, y_true = make_blobs(n_samples=100, centers=4, n_features=2, 
                        random_state=42, cluster_std=0.8)

# 2. Apply agglomerative hierarchical clustering
hc = HierarchicalClustering(n_clusters=4, linkage='complete', method='agglomerative')
labels = hc.fit_predict(X)

# 3. Display the results
print(f"Number of clusters found : {len(set(labels))}")
print(f"Labels : {labels}")

# 4. Visualize
hc.plot_clusters(X)

### Choosing the right `linkage` parameter

Each linkage criterion can lead to a different clustering result. The table below gives a practical guide:


In [ ]:
from notebooks.clustering.utils import plot_hierarchical_linkage_comparison

# Compare the three criteria on the same data
linkages = ['single', 'complete', 'average']
results = []

for link in linkages:
    hc = HierarchicalClustering(n_clusters=4, linkage=link, method='agglomerative')
    labels = hc.fit_predict(X)
    sil = calculate_silhouette(X, labels)

    results.append({
        'linkage': link,
        'labels': labels,
        'silhouette': sil,
    })

plot_hierarchical_linkage_comparison(X, results)

print("Summary:")
print("  - 'single'   : Good for non-convex shapes, sensitive to outliers")
print("  - 'complete' : Compact and balanced clusters, robust to outliers")
print("  - 'average'  : Balanced compromise, usually recommended")

### When to use agglomerative or divisive clustering

| Situation | Recommendation | Reason |
|-----------|---|---|
| You have fewer than 1,000 samples | **Agglomerative** | Usually fast enough, and the dendrogram is available |
| You have more than 10,000 samples | **Divisive** | Can be more memory-friendly depending on the split strategy |
| You do not know the number of clusters | **Agglomerative** | The dendrogram helps guide the choice |
| The clusters are roughly spherical | **Either** | Both approaches can work well |
| The clusters have irregular shapes | **Single linkage** | It can capture more complex shapes, with care for outliers |


## 9. Real-life applications

### 1. **Biology and genetics**
Hierarchical dendrograms are used to build **phylogenetic trees**, where species are grouped according to genetic similarity. Each leaf represents a species, and the merge height reflects evolutionary distance.

### 2. **Marketing and customer segmentation**
Companies can use hierarchical clustering to segment customers by purchase behavior, demographics, or engagement level. The dendrogram helps explore segmentation at different levels of detail.

### 3. **Natural language processing**
In document and text analysis, hierarchical clustering can be used to:
- Group similar documents
- Build topic hierarchies, such as sports -> football -> teams
- Study how themes evolve over time

### 4. **Medical imaging**
Medical scans such as MRI, CT, or X-ray images can be grouped by symptom severity, image similarity, or detected disease patterns.

### 5. **Ecology**
Hierarchical clustering can group species by behavior, habitat, diet, or environmental measurements, revealing natural structures without imposing labels in advance.


## 10. Key takeaways

**Advantages of hierarchical clustering:**
- The number of clusters does not have to be fixed upfront
- The dendrogram gives a complete hierarchical view
- Agglomerative clustering is deterministic and reproducible
- Single linkage can detect non-convex structures

**Important limitations:**
- $O(n^3)$ complexity can be slow on large datasets
- Merges are final; the algorithm does not backtrack
- Results are sensitive to the linkage criterion
- Dendrograms become hard to read visually when there are many samples

**Best use cases:**
- Small to medium-sized datasets
- Exploratory analysis where several values of $k$ must be compared
- Naturally hierarchical structures such as taxonomy or phylogeny
- Visualization and interpretation of data structure


## 11. References

### Articles and resources
- **Hierarchical Clustering** - Introduction to Statistical Learning (ISLR)
  https://www.statlearning.com/
  
- **Agglomerative Hierarchical Clustering Algorithms** - Müllner, 2011
  https://arxiv.org/pdf/1109.2378.pdf
  
- **An Introduction to Hierarchical Clustering** - StatQuest with Josh Starmer (YouTube)
  https://youtu.be/7xHsRkOdVKc

### Related libraries
- **SciPy**: `scipy.cluster.hierarchy` - reference implementation with advanced dendrogram tools
- **scikit-learn**: `AgglomerativeClustering` - scikit-learn implementation
- **seaborn**: `clustermap()` - dendrogram with an integrated heatmap

### Datasets for practice
- **Iris**: classic small dataset, ideal for learning
- **Digits**: handwritten digit images, 1,797 samples and 64 features
- **Wine**: wine classification dataset, 178 samples and 13 features
- **Custom**: generated datasets with `sklearn.datasets.make_blobs()` or `make_moons()`
